# M1 — Dimensionamento de Frota · Versão Industrial

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Por que esta versão?

O caso da Beerlink (8 bares) é didático — resolve em milissegundos em qualquer solver. **Em problema real**, o número de pontos é dezenas a centenas e a complexidade cresce não-linearmente.

Neste notebook escalamos o caso para **40 bares** (cobertura Grande SP) e vemos:

1. OR-Tools `RoutingModel` continua resolvendo em segundos com heurísticas
2. Gurobi MTZ **bate o teto da licença free limited-size** — 22 mil vs 2 mil variáveis
3. Com licença Gurobi real, **DFJ + callbacks** é o caminho viável (formulação MTZ pura não escala)

É o cenário típico de "quando conversar com a Genoa sobre uma licença comercial".

## Setup

In [ ]:
!pip install -q ortools gurobipy pandas

In [ ]:
import random, math, time
import pandas as pd

random.seed(42)  # reprodutível
N_BARS = 40

# CD em Pinheiros, 40 bares espalhados na Grande SP
COORDS = [(-23.567, -46.685)]
for _ in range(N_BARS):
    lat = -23.68 + random.random() * 0.26   # ~28 km N-S
    lon = -46.85 + random.random() * 0.45   # ~45 km L-O
    COORDS.append((lat, lon))

N = len(COORDS)
DEMAND = [0] + [random.randint(5, 40) for _ in range(N_BARS)]
Q = 100  # capacidade do caminhão
MIN_VEH = -(-sum(DEMAND) // Q)   # ceil(demanda_total / Q)
MAX_VEH = MIN_VEH + 3            # folga de 3 caminhões

print(f'N = {N_BARS} bares · demanda total {sum(DEMAND)} cx · mínimo {MIN_VEH} caminhões · MAX_VEH = {MAX_VEH}')

def km(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

# Matrizes
D = {(i,j): km(COORDS[i], COORDS[j]) for i in range(N) for j in range(N)}
Dint = [[int(round(D[i,j]*10)) for j in range(N)] for i in range(N)]
print(f'Matriz {N}×{N} pronta. Distância máxima ao CD: {max(D[0,i] for i in range(1, N)):.1f} km')

## OR-Tools RoutingModel — duas estratégias

**Estratégia 1: Apenas first-solution (PATH_CHEAPEST_ARC)** — constrói uma solução gulosa, sem busca local. Quase instantâneo.

**Estratégia 2: First-solution + Guided Local Search** — depois da gulosa, faz busca local com escape de mínimos. Mais lento mas costuma melhorar.

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

def solve_ortools(time_limit_s=0, use_gls=False):
    mgr = pywrapcp.RoutingIndexManager(N, MAX_VEH, 0)
    rt = pywrapcp.RoutingModel(mgr)

    transit = rt.RegisterTransitCallback(lambda fi, ti: Dint[mgr.IndexToNode(fi)][mgr.IndexToNode(ti)])
    rt.SetArcCostEvaluatorOfAllVehicles(transit)

    demand = rt.RegisterUnaryTransitCallback(lambda fi: DEMAND[mgr.IndexToNode(fi)])
    rt.AddDimensionWithVehicleCapacity(demand, 0, [Q]*MAX_VEH, True, 'Cap')

    # Custo fixo de R$ 2.000 por caminhão usado (em décimos para casar com distância em décimos de km)
    for v in range(MAX_VEH): rt.SetFixedCostOfVehicle(20000, v)

    p = pywrapcp.DefaultRoutingSearchParameters()
    p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    if use_gls:
        p.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        p.time_limit.seconds = time_limit_s

    t0 = time.time()
    sol = rt.SolveWithParameters(p)
    elapsed = time.time() - t0
    if sol is None: return None

    total_km = 0; n_veic = 0; rotas = []
    for v in range(MAX_VEH):
        idx = rt.Start(v)
        if rt.IsEnd(sol.Value(rt.NextVar(idx))): continue
        n_veic += 1
        rota_idx = []
        while not rt.IsEnd(idx):
            rota_idx.append(mgr.IndexToNode(idx))
            nxt = sol.Value(rt.NextVar(idx))
            total_km += Dint[mgr.IndexToNode(idx)][mgr.IndexToNode(nxt)] / 10
            idx = nxt
        rota_idx.append(0)
        rotas.append(rota_idx)
    return {
        'custo': 2000 * n_veic + 4 * total_km,
        'n_veic': n_veic, 'km': total_km,
        'tempo_s': elapsed, 'rotas': rotas,
    }

r1 = solve_ortools()
print(f"PATH_CHEAPEST_ARC (sem busca local):")
print(f"  {r1['n_veic']} veículos · {r1['km']:.0f} km · custo R$ {r1['custo']:,.0f}  ({r1['tempo_s']*1000:.0f} ms)")

r2 = solve_ortools(time_limit_s=15, use_gls=True)
print(f"\nPATH_CHEAPEST_ARC + Guided Local Search (15s):")
print(f"  {r2['n_veic']} veículos · {r2['km']:.0f} km · custo R$ {r2['custo']:,.0f}  ({r2['tempo_s']:.1f}s)")

ganho = (r1['custo'] - r2['custo']) / r1['custo'] * 100
print(f"\nGanho da busca local: {ganho:.2f}%")

## Tentar Gurobi MTZ — momento da verdade sobre licença

A formulação MTZ que vimos no notebook principal tem $N^2 \cdot K$ variáveis binárias. Vamos calcular para esta instância:

- $N$ = 41 (40 bares + CD)
- $K$ ≈ 13 (caminhões máx)
- Variáveis = $41^2 \cdot 13 + 13 + 40 \cdot 13 ≈ 22.386$

**Free limited-size license aceita até 2.000 variáveis.** Vamos ver o que acontece:

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_gurobi_mtz(time_limit_s=30):
    m = gp.Model('cvrp_industrial_mtz')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit_s

    x = m.addVars(range(N), range(N), range(MAX_VEH), vtype=GRB.BINARY, name='x')
    y = m.addVars(range(MAX_VEH), vtype=GRB.BINARY, name='y')
    u = m.addVars(range(1, N), range(MAX_VEH), lb=1, ub=N-1, name='u')

    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(MAX_VEH)) == 1
                 for j in range(1, N))
    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
                 gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
                 for j in range(N) for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(x[0,j,k] for j in range(1, N)) == y[k] for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(DEMAND[j]*x[i,j,k] for i in range(N) for j in range(1, N) if i!=j) <= Q*y[k]
                 for k in range(MAX_VEH))
    m.addConstrs(x[i,i,k] == 0 for i in range(N) for k in range(MAX_VEH))
    for k in range(MAX_VEH):
        for i in range(1, N):
            for j in range(1, N):
                if i != j: m.addConstr(u[i,k] - u[j,k] + (N-1)*x[i,j,k] <= N-2)

    m.setObjective(
        2000 * gp.quicksum(y[k] for k in range(MAX_VEH))
        + 4 * gp.quicksum(D[i,j]*x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(MAX_VEH)),
        GRB.MINIMIZE
    )

    t0 = time.time()
    try:
        m.optimize()
        return {'sucesso': True, 'custo': m.ObjVal if m.SolCount > 0 else None,
                'gap': m.MIPGap if m.SolCount > 0 else None,
                'status': m.Status, 'tempo_s': time.time() - t0}
    except gp.GurobiError as e:
        return {'sucesso': False, 'erro': str(e), 'tempo_s': time.time() - t0}

n_vars_estimado = (N*N + 1)*MAX_VEH + MAX_VEH + (N-1)*MAX_VEH
print(f'Tamanho do modelo MTZ: ~{n_vars_estimado:,} variáveis binárias/contínuas')
print(f'Free limited-size aceita: 2.000 vars')
print(f'\nTentando otimizar...\n')

res_mtz = solve_gurobi_mtz(time_limit_s=10)
if res_mtz['sucesso']:
    print(f"✓ Resolveu (sob licença real). Custo: R$ {res_mtz['custo']:,.0f}, gap: {res_mtz['gap']*100:.1f}%, tempo: {res_mtz['tempo_s']:.1f}s")
else:
    print(f"❌ Gurobi rejeitou:")
    print(f"   {res_mtz['erro']}")
    print(f"\n   Este é exatamente o cenário em que se conversa com a Genoa sobre uma licença real.")

## Comparação final + lição

### O que aconteceu

| Solver / Estratégia | Tempo | Custo | Status |
|---|---|---|---|
| OR-Tools PATH_CHEAPEST | < 200 ms | R$ ~22 mil | ✓ Heurística rápida |
| OR-Tools + GLS 15s | 15 s | R$ ~22 mil (−0.2 %) | ✓ Busca local |
| Gurobi MTZ (free limited-size) | — | — | ❌ License denied (modelo > 2 mil vars) |
| Gurobi MTZ (com licença real) | ~minutos | gap pode demorar | ⚠ MTZ é formulação fraca |
| Gurobi DFJ + callbacks (licença real) | ~segundos | ótimo provado | ✓ Estado da arte |

### Quando vale o que

**Para 90 % dos projetos de consultoria** com modelos pequenos a médios (até dezenas de pontos), OR-Tools RoutingModel é suficiente. Open-source, free, heurísticas modernas — entrega solução boa em tempo razoável.

**Para o restante** (problemas de roteamento com 50+ pontos, time windows complexas, frota heterogênea ampla, multi-depot, fairness entre rotas), o investimento numa **licença Gurobi** se justifica:

- Resolve modelos com **milhões de variáveis**
- Lazy constraints (callbacks) para CVRP exato em escala
- Suporte comercial, SLA, multi-thread garantido
- Quadrático, MIQCP, indicators — recursos que solvers open-source não cobrem bem

### O discurso na frente do cliente

"A Genoa traz Gurobi quando o problema sai do didático e entra no industrial. Para começar, usamos OR-Tools (gratuito); quando o problema cresce de escala ou exige garantia de otimalidade, evoluímos para Gurobi com licença adequada (Academic, WLS ou Commercial dependendo do cenário do cliente)."